# Практична робота: AES-ECB, AES-CBC, AES-CFB та моделювання атак

У цьому ноутбуці реалізовано режими `AES-ECB`, `AES-CBC`, `AES-CFB` (CFB-128), перевірку на тестових векторах, аналіз генератора випадкових даних та моделювання двох атак (на CBC і CFB).

Стандарт алгоритму: **FIPS 197** (AES, блок 128 біт, ключ 128/192/256 біт).

## 0. Залежності

Якщо пакет `pycryptodome` відсутній, розкоментуйте наступний рядок.

In [3]:
 %pip install pycryptodome

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 13.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from __future__ import annotations

from Crypto.Cipher import AES
from secrets import token_bytes
import random

BLOCK_SIZE = 16  # AES block size in bytes


def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))


def split_blocks(data: bytes, block_size: int = BLOCK_SIZE) -> list[bytes]:
    if len(data) % block_size != 0:
        raise ValueError("Data length must be a multiple of block size.")
    return [data[i:i + block_size] for i in range(0, len(data), block_size)]


def pkcs7_pad(data: bytes, block_size: int = BLOCK_SIZE) -> bytes:
    pad_len = block_size - (len(data) % block_size)
    return data + bytes([pad_len] * pad_len)


def pkcs7_unpad(data: bytes, block_size: int = BLOCK_SIZE) -> bytes:
    if not data or len(data) % block_size != 0:
        raise ValueError("Invalid padded data length.")
    pad_len = data[-1]
    if pad_len < 1 or pad_len > block_size:
        raise ValueError("Invalid padding value.")
    if data[-pad_len:] != bytes([pad_len] * pad_len):
        raise ValueError("Invalid PKCS#7 padding bytes.")
    return data[:-pad_len]


def aes_block_encrypt(block: bytes, key: bytes) -> bytes:
    if len(block) != BLOCK_SIZE:
        raise ValueError("AES block must be 16 bytes.")
    return AES.new(key, AES.MODE_ECB).encrypt(block)


def aes_block_decrypt(block: bytes, key: bytes) -> bytes:
    if len(block) != BLOCK_SIZE:
        raise ValueError("AES block must be 16 bytes.")
    return AES.new(key, AES.MODE_ECB).decrypt(block)


def ecb_encrypt_raw(plaintext: bytes, key: bytes) -> bytes:
    blocks = split_blocks(plaintext, BLOCK_SIZE)
    return b"".join(aes_block_encrypt(b, key) for b in blocks)


def ecb_decrypt_raw(ciphertext: bytes, key: bytes) -> bytes:
    blocks = split_blocks(ciphertext, BLOCK_SIZE)
    return b"".join(aes_block_decrypt(b, key) for b in blocks)


def cbc_encrypt_raw(plaintext: bytes, key: bytes, iv: bytes) -> bytes:
    if len(iv) != BLOCK_SIZE:
        raise ValueError("IV must be 16 bytes for CBC.")
    blocks = split_blocks(plaintext, BLOCK_SIZE)
    prev = iv
    out = []
    for block in blocks:
        x = xor_bytes(block, prev)
        c = aes_block_encrypt(x, key)
        out.append(c)
        prev = c
    return b"".join(out)


def cbc_decrypt_raw(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    if len(iv) != BLOCK_SIZE:
        raise ValueError("IV must be 16 bytes for CBC.")
    blocks = split_blocks(ciphertext, BLOCK_SIZE)
    prev = iv
    out = []
    for block in blocks:
        p = xor_bytes(aes_block_decrypt(block, key), prev)
        out.append(p)
        prev = block
    return b"".join(out)


def cfb_encrypt_raw(plaintext: bytes, key: bytes, iv: bytes) -> bytes:
    if len(iv) != BLOCK_SIZE:
        raise ValueError("IV must be 16 bytes for CFB-128.")
    prev = iv
    out = []
    for i in range(0, len(plaintext), BLOCK_SIZE):
        chunk = plaintext[i:i + BLOCK_SIZE]
        ks = aes_block_encrypt(prev, key)
        c = xor_bytes(chunk, ks[:len(chunk)])
        out.append(c)
        prev = c if len(c) == BLOCK_SIZE else prev
    return b"".join(out)


def cfb_decrypt_raw(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    if len(iv) != BLOCK_SIZE:
        raise ValueError("IV must be 16 bytes for CFB-128.")
    prev = iv
    out = []
    for i in range(0, len(ciphertext), BLOCK_SIZE):
        chunk = ciphertext[i:i + BLOCK_SIZE]
        ks = aes_block_encrypt(prev, key)
        p = xor_bytes(chunk, ks[:len(chunk)])
        out.append(p)
        prev = chunk if len(chunk) == BLOCK_SIZE else prev
    return b"".join(out)


def ecb_encrypt(plaintext: bytes, key: bytes) -> bytes:
    return ecb_encrypt_raw(pkcs7_pad(plaintext), key)


def ecb_decrypt(ciphertext: bytes, key: bytes) -> bytes:
    return pkcs7_unpad(ecb_decrypt_raw(ciphertext, key))


def cbc_encrypt(plaintext: bytes, key: bytes, iv: bytes) -> bytes:
    return cbc_encrypt_raw(pkcs7_pad(plaintext), key, iv)


def cbc_decrypt(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    return pkcs7_unpad(cbc_decrypt_raw(ciphertext, key, iv))

## 1. Перевірка на тестових векторах (NIST SP 800-38A)

In [8]:
# NIST SP 800-38A test vectors (AES-128)
key = bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c")
iv = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
pt = bytes.fromhex(
    "6bc1bee22e409f96e93d7e117393172a"
    "ae2d8a571e03ac9c9eb76fac45af8e51"
    "30c81c46a35ce411e5fbc1191a0a52ef"
    "f69f2445df4f9b17ad2b417be66c3710"
)

ecb_expected = bytes.fromhex(
    "3ad77bb40d7a3660a89ecaf32466ef97"
    "f5d3d58503b9699de785895a96fdbaaf"
    "43b1cd7f598ece23881b00e3ed030688"
    "7b0c785e27e8ad3f8223207104725dd4"
)
cbc_expected = bytes.fromhex(
    "7649abac8119b246cee98e9b12e9197d"
    "5086cb9b507219ee95db113a917678b2"
    "73bed6b8e3c1743b7116e69e22229516"
    "3ff1caa1681fac09120eca307586e1a7"
)
cfb_expected = bytes.fromhex(
    "3b3fd92eb72dad20333449f8e83cfb4a"
    "c8a64537a0b3a93fcde3cdad9f1ce58b"
    "26751f67a3cbb140b1808cf187a4f4df"
    "c04b05357c5d1c0eeac4c66f9ff7f2e6"
)

ecb_got = ecb_encrypt_raw(pt, key)
cbc_got = cbc_encrypt_raw(pt, key, iv)
cfb_got = cfb_encrypt_raw(pt, key, iv)

print("[NIST] Input preview")
print("PT[0] =", split_blocks(pt)[0].hex())
print("IV    =", iv.hex())

print("\n[NIST] Block-by-block compare")
for mode_name, got, expected in [
    ("ECB", ecb_got, ecb_expected),
    ("CBC", cbc_got, cbc_expected),
    ("CFB", cfb_got, cfb_expected),
]:
    got_blocks = split_blocks(got)
    exp_blocks = split_blocks(expected)
    print(f"{mode_name}:")
    for i, (g, e) in enumerate(zip(got_blocks, exp_blocks), start=1):
        print(f"  block {i}: match={g == e}")
        print(f"    got={g.hex()}")
        print(f"    exp={e.hex()}")

assert ecb_got == ecb_expected, "ECB vector mismatch"
assert cbc_got == cbc_expected, "CBC vector mismatch"
assert cfb_got == cfb_expected, "CFB vector mismatch"

assert ecb_decrypt_raw(ecb_got, key) == pt
assert cbc_decrypt_raw(cbc_got, key, iv) == pt
assert cfb_decrypt_raw(cfb_got, key, iv) == pt

print("\nAll NIST vectors passed for ECB/CBC/CFB (AES-128).")

[NIST] Input preview
PT[0] = 6bc1bee22e409f96e93d7e117393172a
IV    = 000102030405060708090a0b0c0d0e0f

[NIST] Block-by-block compare
ECB:
  block 1: match=True
    got=3ad77bb40d7a3660a89ecaf32466ef97
    exp=3ad77bb40d7a3660a89ecaf32466ef97
  block 2: match=True
    got=f5d3d58503b9699de785895a96fdbaaf
    exp=f5d3d58503b9699de785895a96fdbaaf
  block 3: match=True
    got=43b1cd7f598ece23881b00e3ed030688
    exp=43b1cd7f598ece23881b00e3ed030688
  block 4: match=True
    got=7b0c785e27e8ad3f8223207104725dd4
    exp=7b0c785e27e8ad3f8223207104725dd4
CBC:
  block 1: match=True
    got=7649abac8119b246cee98e9b12e9197d
    exp=7649abac8119b246cee98e9b12e9197d
  block 2: match=True
    got=5086cb9b507219ee95db113a917678b2
    exp=5086cb9b507219ee95db113a917678b2
  block 3: match=True
    got=73bed6b8e3c1743b7116e69e22229516
    exp=73bed6b8e3c1743b7116e69e22229516
  block 4: match=True
    got=3ff1caa1681fac09120eca307586e1a7
    exp=3ff1caa1681fac09120eca307586e1a7
CFB:
  block 1: match=Tr

## 2. Приклад використання з випадковими ключем та IV

In [9]:
message = b"Konfidentsiini dany ta navchalnyi pryklad AES modes."
key256 = token_bytes(32)  # AES-256 key
iv_rand = token_bytes(16)

ct_ecb = ecb_encrypt(message, key256)
ct_cbc = cbc_encrypt(message, key256, iv_rand)
ct_cfb = cfb_encrypt_raw(message, key256, iv_rand)

pt_ecb = ecb_decrypt(ct_ecb, key256)
pt_cbc = cbc_decrypt(ct_cbc, key256, iv_rand)
pt_cfb = cfb_decrypt_raw(ct_cfb, key256, iv_rand)

assert pt_ecb == message
assert pt_cbc == message
assert pt_cfb == message

print("Message length:", len(message), "bytes")
print("Key (first 16 bytes):", key256[:16].hex())
print("IV:", iv_rand.hex())
print("Ciphertext lengths:")
print("  ECB:", len(ct_ecb), "bytes")
print("  CBC:", len(ct_cbc), "bytes")
print("  CFB:", len(ct_cfb), "bytes")
print("Ciphertext first 16 bytes:")
print("  ECB:", ct_ecb[:16].hex())
print("  CBC:", ct_cbc[:16].hex())
print("  CFB:", ct_cfb[:16].hex())
print("Roundtrip checks:")
print("  ECB:", pt_ecb == message)
print("  CBC:", pt_cbc == message)
print("  CFB:", pt_cfb == message)
print("ECB/CBC/CFB encryption-decryption roundtrip: OK")

Message length: 52 bytes
Key (first 16 bytes): 09636c9e7966fd7b959ca0b6f0e2d4cc
IV: f6c863f2ce8524c074c4dde274f20f70
Ciphertext lengths:
  ECB: 64 bytes
  CBC: 64 bytes
  CFB: 52 bytes
Ciphertext first 16 bytes:
  ECB: c4c58310944bf7c7d39f63df5613ac19
  CBC: 097d46f469558d5f1ca6225bace58865
  CFB: f5c7ab8f3649f58198cc551c6a579d5a
Roundtrip checks:
  ECB: True
  CBC: True
  CFB: True
ECB/CBC/CFB encryption-decryption roundtrip: OK


## 3. Аналіз генератора випадкових даних

Для ключів/IV використано `secrets.token_bytes()`. Це CSPRNG-інтерфейс у Python, який опирається на ОС-джерело ентропії (`os.urandom`, у Windows — системний крипто-API).

Порівняємо з `random.Random`, який **не є криптостійким** і відтворюється за seed.

In [10]:
seed = 12345
r1 = random.Random(seed)
r2 = random.Random(seed)

pseudo1 = bytes(r1.getrandbits(8) for _ in range(16))
pseudo2 = bytes(r2.getrandbits(8) for _ in range(16))

secure1 = token_bytes(16)
secure2 = token_bytes(16)

print("Seed:", seed)
print("random.Random sample #1:", pseudo1.hex())
print("random.Random sample #2:", pseudo2.hex())
print("secrets.token_bytes sample #1:", secure1.hex())
print("secrets.token_bytes sample #2:", secure2.hex())
print("random.Random deterministic with same seed:", pseudo1 == pseudo2)
print("secrets.token_bytes repeat by chance (practically no):", secure1 == secure2)

Seed: 12345
random.Random sample #1: 6abb02d1d3cd4cda5eee3145906f295f
random.Random sample #2: 6abb02d1d3cd4cda5eee3145906f295f
secrets.token_bytes sample #1: df7ba236449330ef27a730474b814809
secrets.token_bytes sample #2: 8cbf1bdbcc2baac25e353472ae8ebc21
random.Random deterministic with same seed: True
secrets.token_bytes repeat by chance (practically no): False


## 4. Атака 1: CBC bit-flipping (модифікація IV)

Ідея: у CBC перший блок plaintext після розшифрування XOR-иться з IV. Якщо змінити байт IV, передбачувано зміниться відповідний байт першого блоку plaintext без знання ключа.

In [11]:
cbc_key = token_bytes(16)
orig_iv = bytearray(token_bytes(16))

# Exactly one block: target symbol '0' at index 6
plaintext_block = b"admin=0;uid=10;X"
cipher = cbc_encrypt_raw(plaintext_block, cbc_key, bytes(orig_iv))

idx = plaintext_block.index(b"0")
delta = ord("0") ^ ord("1")
forged_iv = bytearray(orig_iv)
orig_iv_byte = forged_iv[idx]
forged_iv[idx] ^= delta
new_iv_byte = forged_iv[idx]

decrypted_original = cbc_decrypt_raw(cipher, cbc_key, bytes(orig_iv))
decrypted_forged = cbc_decrypt_raw(cipher, cbc_key, bytes(forged_iv))

print("Target index in block:", idx)
print("Bitflip delta (hex):", hex(delta))
print("IV original:", bytes(orig_iv).hex())
print("IV forged  :", bytes(forged_iv).hex())
print("Changed IV byte:", hex(orig_iv_byte), "->", hex(new_iv_byte))
print("Ciphertext block:", cipher.hex())
print("Original plaintext :", decrypted_original)
print("Forged plaintext   :", decrypted_forged)
print("Attack success     :", b"admin=1" in decrypted_forged)

Target index in block: 6
Bitflip delta (hex): 0x1
IV original: c49a4f2de3903472cf4b2a23358c6f9d
IV forged  : c49a4f2de3903572cf4b2a23358c6f9d
Changed IV byte: 0x34 -> 0x35
Ciphertext block: 399a83102c0558289f4eda22eaee5091
Original plaintext : b'admin=0;uid=10;X'
Forged plaintext   : b'admin=1;uid=10;X'
Attack success     : True


## 5. Атака 2: CFB IV reuse (known-plaintext recovery)

Якщо в CFB повторно використати той самий `IV` з тим самим ключем, перший блок працює з однаковим keystream.
Тоді для перших блоків виконується: `C1_a XOR C1_b = P1_a XOR P1_b`.

Якщо attacker знає `P1_a` і має `C1_a, C1_b`, він може відновити `P1_b` без знання ключа.
Це вже інший клас атаки: **витік конфіденційності через повторне використання IV**, а не керована модифікація bit-flipping.

In [14]:
cfb_key = token_bytes(16)
reused_iv = token_bytes(16)

# Two different plaintexts encrypted under the same key+IV (unsafe)
msg_a = b"known_header=USER;balance=1000;"
msg_b = b"secret_hdr=VIP!;balance=9000;"

ct_a = cfb_encrypt_raw(msg_a, cfb_key, reused_iv)
ct_b = cfb_encrypt_raw(msg_b, cfb_key, reused_iv)

# Attacker knows msg_a and sees ct_a, ct_b. Recover first block of msg_b.
p1_a = msg_a[:BLOCK_SIZE]
p1_b_real = msg_b[:BLOCK_SIZE]
c1_a = ct_a[:BLOCK_SIZE]
c1_b = ct_b[:BLOCK_SIZE]

recovered_p1_b = xor_bytes(xor_bytes(c1_b, c1_a), p1_a)

print("CFB IV reuse demo (same key + same IV)")
print("IV:", reused_iv.hex())
print("BLOCK_SIZE:", BLOCK_SIZE)
print("msg_a:", msg_a)
print("msg_b:", msg_b)
print()
print("First-block data (hex):")
print("  P1_a:", p1_a.hex())
print("  C1_a:", c1_a.hex())
print("  C1_b:", c1_b.hex())
print("  Recovered P1_b:", recovered_p1_b.hex())
print("  Real      P1_b:", p1_b_real.hex())
print()
print("Recovered first block bytes:", recovered_p1_b)
print("Real first block bytes     :", p1_b_real)
print("Attack success:", recovered_p1_b == p1_b_real)

CFB IV reuse demo (same key + same IV)
IV: 5aeed6037876b70de96f7e0eea5d5c21
BLOCK_SIZE: 16
msg_a: b'known_header=USER;balance=1000;'
msg_b: b'secret_hdr=VIP!;balance=9000;'

First-block data (hex):
  P1_a: 6b6e6f776e5f6865616465723d555345
  C1_a: 9b59bdbec9783fb4adf5bd562b07ab9e
  C1_b: 8352b1bbc25308b9a8e3e5725f02d9e0
  Recovered P1_b: 7365637265745f6864723d564950213b
  Real      P1_b: 7365637265745f6864723d564950213b

Recovered first block bytes: b'secret_hdr=VIP!;'
Real first block bytes     : b'secret_hdr=VIP!;'
Attack success: True
